In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# import models that i'll use in
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [28]:
# load the dataset
path_to_datasets = Path().cwd().parent / "dataset"
train_df = pd.read_csv(path_to_datasets / "train_clean.csv")
test_df = pd.read_csv(path_to_datasets / "test_clean.csv")

In [29]:
# define base models with some suggested hyperparameters
base_models = {
    "xgb": XGBClassifier(
        n_estimators=1200, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        tree_method="hist", # hist for faster training as it will use histogram binning instead of training on each individual data point
        device="cuda",  # use cuda to train it on gpu for faster training if available
        random_state=42, n_jobs=-1, # n_jobs=-1 to use all available CPU cores for training
    ),
    "lgbm": LGBMClassifier(
        n_estimators=1500, learning_rate=0.03, num_leaves=64,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbosity=-1, class_weight="balanced"
    ),
    "catboost": CatBoostClassifier(
        iterations=2000, learning_rate=0.05, depth=6,
        loss_function="MultiClass", random_seed=42, verbose=0,
    ),
    "rf": RandomForestClassifier(
        n_estimators=800, min_samples_leaf=2,
        n_jobs=-1, random_state=42
    ),
    'LogisticRegression': Pipeline([
        ('scaler', RobustScaler()),
        ('lr', LogisticRegression(
            C=1.0, max_iter=2000,
            random_state=42, class_weight="balanced"
        ))
    ])
}

meta_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

In [30]:
X = train_df.drop(columns=["status_group"])
y = train_df["status_group"]

In [31]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [32]:
from scipy.stats import entropy

def build_meta(prob_block, n_models, n_classes):
    P = prob_block.reshape(prob_block.shape[0], n_models, n_classes)
    sorted_P = np.sort(P, axis=2)
    feats = [
        prob_block,                          # raw probabilities
        P.max(axis=2),                       # confidence per model
        sorted_P[:, :, -1] - sorted_P[:, :, -2],   # top-2 margin per model
        entropy(P + 1e-9, axis=2),           # uncertainty per model
        P.mean(axis=1),                      # ensemble mean prob per class
        P.std(axis=1),                       # disagreement per class  <-- key signal
    ]
    return np.hstack([f if f.ndim > 1 else f[:, None] for f in feats])

In [33]:
n_splits = 5
n_models = len(base_models)
n_classes = int(y_train.nunique())

label_encoder = LabelEncoder()
label_encoder.fit(y)
y_train = pd.Series(label_encoder.transform(y_train), index=y_train.index)  # type: ignore
y_val   = pd.Series(label_encoder.transform(y_val),   index=y_val.index)    # type: ignore

# align competition test set to the exact training feature columns (drops id / extras)
X_test = test_df[X_train.columns]

meta_features      = np.zeros((X_train.shape[0], n_models * n_classes))
meta_val_features  = np.zeros((X_val.shape[0],   n_models * n_classes))
meta_test_features = np.zeros((X_test.shape[0],  n_models * n_classes))

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for model_index, (model_name, model) in enumerate(base_models.items()):
    print(f"Training base model {model_index + 1}/{n_models}: {model_name}")

    fold_val_preds  = np.zeros((X_val.shape[0],  n_splits, n_classes))
    fold_test_preds = np.zeros((X_test.shape[0], n_splits, n_classes))
    cols = slice(model_index * n_classes, (model_index + 1) * n_classes)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        model.fit(X_fold_train, y_fold_train)

        meta_features[val_idx, cols] = model.predict_proba(X_fold_val)
        fold_val_preds[:, fold, :]   = model.predict_proba(X_val)
        fold_test_preds[:, fold, :]  = model.predict_proba(X_test)

    meta_val_features[:, cols]  = fold_val_preds.mean(axis=1)
    meta_test_features[:, cols] = fold_test_preds.mean(axis=1)
    print(f"Completed training for base model: {model_name}")

meta_features      = build_meta(meta_features,      n_models, n_classes)
meta_val_features  = build_meta(meta_val_features,  n_models, n_classes)
meta_test_features = build_meta(meta_test_features, n_models, n_classes)

print("Meta-features built for train / val / test.")

Training base model 1/5: xgb
Completed training for base model: xgb
Training base model 2/5: lgbm
Completed training for base model: lgbm
Training base model 3/5: catboost
Completed training for base model: catboost
Training base model 4/5: rf
Completed training for base model: rf
Training base model 5/5: LogisticRegression
Completed training for base model: LogisticRegression
Meta-features built for train / val / test.


In [34]:
print(f"Now Training the meta model {meta_model.__class__.__name__}....")

# Train the meta-model on the out-of-fold predictions from the base models
meta_model.fit(meta_features, y_train)

val_preds = meta_model.predict(meta_val_features)
print(classification_report(y_val, val_preds))
print(accuracy_score(y_val, val_preds))

Now Training the meta model LogisticRegression....
              precision    recall  f1-score   support

           0       0.79      0.91      0.85      6364
           1       0.64      0.23      0.34       838
           2       0.85      0.77      0.80      4523

    accuracy                           0.81     11725
   macro avg       0.76      0.64      0.66     11725
weighted avg       0.80      0.81      0.79     11725

0.8063965884861407


In [35]:
final_preds = label_encoder.inverse_transform(meta_model.predict(meta_test_features))

test_ids = pd.read_csv(path_to_datasets / "test_ids.csv")["id"]
submission_df = pd.DataFrame({
    "id": test_ids,
    "status_group": final_preds
})

assert len(submission_df) == len(test_df)
assert submission_df["status_group"].isnull().sum() == 0
print(submission_df["status_group"].value_counts())
submission_df.to_csv("../submissions/stacking_modelV3.csv", index=False)
submission_df.head()

status_group
functional                 9602
non functional             5023
functional needs repair     225
Name: count, dtype: int64


,id,status_group
0,50785,functional
1,51630,functional
2,17168,functional
3,45559,non functional
4,49871,functional
